In [ ]:
# Step 0: Install required libraries (run once at the start of each lab session)
%pip install --upgrade scikit-learn pandas matplotlib seaborn

In [1]:
# Step 1: Download the Bank Marketing dataset from UCI into the lab
import urllib.request, zipfile, io

url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
outer_zip = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
inner_zip = zipfile.ZipFile(io.BytesIO(outer_zip.read("bank.zip")))
inner_zip.extract("bank-full.csv")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [2]:
# Step 2: Import libraries and load the dataset
import pandas as pd
import numpy as np

df = pd.read_csv("bank-full.csv", sep=";")

print("Shape (rows, columns):", df.shape)
df.head()

/home/cloud/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/home/cloud/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Shape (rows, columns): (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
# Step 3: Understand the data - feature types, missing values, target distribution
print("Data type of each column:")
print(df.dtypes)

print("\nNumber of 'unknown' values in each column (these are missing values):")
print((df == "unknown").sum())

print("\nTarget variable (y) - count of each class:")
print(df["y"].value_counts())

print("\nTarget variable (y) - percentage of each class:")
print(df["y"].value_counts(normalize=True) * 100)

Data type of each column:
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

Number of 'unknown' values in each column (these are missing values):
age              0
job            288
marital          0
education     1857
default          0
balance          0
housing          0
loan             0
contact      13020
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome     36959
y                0
dtype: int64

Target variable (y) - count of each class:
y
no     39922
yes     5289
Name: count, dtype: int64

Target variable (y) - percentage of each class:
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


In [4]:
# Step 4: Handle missing values and duplicates

# (a) Missing values
# 'job' and 'education' have only a few 'unknown' values -> mark them as missing (NaN).
# They will be filled later with the most frequent value, learned ONLY from training data.
df["job"] = df["job"].replace("unknown", np.nan)
df["education"] = df["education"].replace("unknown", np.nan)

# 'contact' (29% unknown) and 'poutcome' (82% unknown) have too many 'unknown' values.
# Here 'unknown' has a real meaning (e.g. customer was never contacted before),
# so we keep 'unknown' as its own category instead of filling it.

print("Missing values (NaN) after marking:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# (b) Duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Missing values (NaN) after marking:
job           288
education    1857
dtype: int64

Number of duplicate rows: 0
Shape after removing duplicates: (45211, 17)


In [5]:
# Step 5: Prepare features (X) and target (y), then split 80:20
from sklearn.model_selection import train_test_split

# Drop 'duration': it is known only AFTER the phone call ends,
# so it cannot be used to predict BEFORE calling (UCI dataset notes advise this).
df = df.drop(columns=["duration"])

# X = all input columns (features), y = target column converted to numbers
X = df.drop(columns=["y"])
y = df["y"].map({"yes": 1, "no": 0})

# 80% training, 20% testing, same class ratio in both, fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:    ", X_test.shape)
print("\nClass % in training set:")
print(y_train.value_counts(normalize=True) * 100)
print("\nClass % in test set:")
print(y_test.value_counts(normalize=True) * 100)

Training set: (36168, 15)
Test set:     (9043, 15)

Class % in training set:
y
0    88.301814
1    11.698186
Name: proportion, dtype: float64

Class % in test set:
y
0    88.300343
1    11.699657
Name: proportion, dtype: float64


In [16]:
# Step 6: Fill missing values, encode categories, scale numbers
# All steps are LEARNED from training data only, then APPLIED to test data
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]
categorical_cols = ["job", "marital", "education", "default", "housing",
                    "loan", "contact", "month", "poutcome"]

# Numbers -> scale them to a similar range
numeric_steps = Pipeline([
    ("scale", StandardScaler())
])

# Text categories -> fill missing with most frequent value, then one-hot encode
categorical_steps = Pipeline([
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_steps, numeric_cols),
    ("cat", categorical_steps, categorical_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)   # learn from TRAIN + apply
X_test_prep = preprocessor.transform(X_test)         # only apply to TEST

print("Training data after preprocessing:", X_train_prep.shape)
print("Test data after preprocessing:    ", X_test_prep.shape)
print("Missing values left in training data:", np.isnan(X_train_prep).sum())

Training data after preprocessing: (36168, 48)
Test data after preprocessing:     (9043, 48)
Missing values left in training data: 0


In [18]:
# Step 7: Save the 20% test split as test_data.csv (required for GitHub)
test_data = X_test.copy()
test_data["y"] = y_test.map({1: "yes", 0: "no"})
test_data.to_csv("test_data.csv", index=False, na_rep="unknown")

print("Saved test_data.csv with shape:", test_data.shape)

Saved test_data.csv with shape: (9043, 16)
